In [22]:
import re
import joblib
from nltk.corpus import stopwords
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
import Stemmer
import pandas as pd
import numpy as np

In [ ]:
svm = joblib.load('training_results/SVend-Martin.joblib')
vectorizer = joblib.load('training_results/SVend-Martin_vect.joblib')

# Uncompiled patterns for polars later
email_str = r'[^ \n,"]+@[^ \n,"]+\.[^ \n,"]+'
date_str = r'[0-9]{2,4}[-/][0-9]{2,4}[-/][0-9]{2,4}'
url_str = r'(?:http)?s?(?://)?[^ \n,"]+\.[a-z]{2,}[^ \n,"]+'
num_str = r'[0-9]+[,.]?[0-9]*'
special_str = r'[^a-zA-Z\-<>\'’‘ʼ′＇]'
apostrephe_str = r'\'’‘ʼ′＇'
ws_str = r'\s+'

# Pattern strings — shared with Task 2 (Polars uses these directly)
email_re = re.compile(email_str)
date_re = re.compile(date_str)
url_re = re.compile(url_str)
num_re = re.compile(num_str)
special_re = re.compile(special_str)
apostrephe_re = re.compile(apostrephe_str)
ws_re = re.compile(ws_str)

# Initialize NLTK Stop words and PyStemmer
stop_words = set(stopwords.words("english"))
stemmer = Stemmer.Stemmer("english")

def preprocess_text(doc: str):
    # Cleaning
    lower_case = doc.lower()
    substituted = email_re.sub(" <EMAIL> ", lower_case)
    substituted = date_re.sub(" <DATE> ", substituted)
    substituted = url_re.sub(" <URL> ", substituted)
    substituted = num_re.sub(" <NUM> ", substituted)
    no_specials = special_re.sub(" ", substituted)
    no_apostrophes = apostrephe_re.sub("", no_specials)
    cleaned = ws_re.sub(' ', no_apostrophes).strip()

    # Stemming and Filtering
    words = [w for w in cleaned.split() if w not in stop_words]
    return " ".join(stemmer.stemWords(words))

In [29]:
# liar dataset
liar_train = pd.read_table('./liar_dataset/train.tsv').iloc[:,[1,2]]
liar_val = pd.read_table('./liar_dataset/valid.tsv').iloc[:,[1,2]]
liar_test = pd.read_table('./liar_dataset/test.tsv').iloc[:,[1,2]]

liar_train.columns = ['isfake','title']
liar_val.columns = ['isfake','title']
liar_test.columns = ['isfake','title']

df = pd.concat([liar_train, liar_val, liar_test], axis=0)
df

df.columns = ['isfake','title']
df['title'] = df['title'].apply(preprocess_text)

real_labels = ['mostly-true', 'true']
fake_labels = ['false','pants-fire', 'half-true', 'barely-true']

df['isfake'] = df['isfake'].apply(lambda x : 0 if x in real_labels else 1)
X, y = df['title'], df['isfake']

X = vectorizer.transform(X)

In [ ]:
# Predictions
y_preds = svm.predict(X)
print(classification_report(y, y_preds))

              precision    recall  f1-score   support

           0       0.41      0.08      0.14      4506
           1       0.65      0.93      0.77      8282

    accuracy                           0.63     12788
   macro avg       0.53      0.51      0.45     12788
weighted avg       0.57      0.63      0.55     12788

